# Libraries

In [ ]:
# 0. Clean slate — remove existing copies before reinstalling
!pip uninstall -y torch torchvision torchaudio torchao

# 3. HuggingFace ecosystem + llmcompressor
!pip install -q -U \
    transformers \
    accelerate \
    datasets \
    huggingface_hub \
    llmcompressor

In [ ]:
import random
import numpy as np
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    set_seed,
)

from llmcompressor import oneshot
from llmcompressor.modifiers.awq import AWQModifier

from huggingface_hub import (
    login,
    create_repo,
    upload_folder,
)

from kaggle_secrets import UserSecretsClient
from datasets import load_dataset

# Global Variables

In [ ]:
BASE_ID = "microsoft/Phi-4-mini-instruct"
MODEL_ID = "EdgeCompress01/Phi-4-mini-instruct-WANDA"
OUTPUT_DIR = "/kaggle/working/Phi-4-mini-instruct-WANDA-AWQ-4bit"
REPO_ID = f"EdgeCompress01/Phi-4-mini-instruct-WANDA-AWQ-4bit"
SEED = 42

# Functions

In [ ]:
# --- 2. REPRODUCIBILITY ---
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to {seed}")

def format_chat(batch):
    texts = [
        tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        for messages in batch["messages"]
    ]
    return {"text": texts}

# Set Seed

In [ ]:
set_reproducibility(SEED)

# Huggingface Login

In [ ]:
# --- 3. HUGGING FACE AUTHENTICATION ---
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logged in")

# Load Model & Tokenizer

In [ ]:
#Model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    subfolder="Models/50",
    device_map="auto",
    dtype=torch.bfloat16
)

#Tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_ID)

# AWQ Quantization

**Configurations**

In [ ]:
recipe = [
    AWQModifier(
        ignore=["lm_head", "embed_tokens"], 
        scheme="W4A16",
        targets=["Linear"]
    )
]

**Calibration Dataset**

In [ ]:
dataset = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft").shuffle(seed=SEED).select(range(64))

formatted_dataset = dataset.map(
    format_chat,
    batched=True,
    remove_columns=dataset.column_names  # keep only "text"
)

**Apply quantization**

In [ ]:
oneshot(
    model=model,
    tokenizer=tokenizer,
    dataset=formatted_dataset,
    text_column="text",
    recipe=recipe,
    max_seq_length=512
)

**Saving**

In [ ]:
model.config._name_or_path = ""

model.save_pretrained(OUTPUT_DIR,skip_sparsity_compression_stats=False)#VERY IMPORTANT
tokenizer.save_pretrained(OUTPUT_DIR)

print("Model saved successfully")

# PUSH TO HUGGING FACE

In [ ]:
create_repo(REPO_ID, repo_type="model", exist_ok=True)

upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=REPO_ID,
    path_in_repo="Sparsity/50"
)